In [2]:
import pandas as pd

In [14]:
eda = pd.read_csv("data/EDA_df.csv").rename(columns={'data':'eda'})

In [15]:
eda

,eda,student,test
0,0.000000,S1,Final
1,0.005125,S1,Final
2,0.020501,S1,Final
3,0.021783,S1,Final
4,0.023064,S1,Final
...,...,...,...
1773115,0.021781,S10,Midterm 2
1773116,0.020500,S10,Midterm 2
1773117,0.020500,S10,Midterm 2
1773118,0.020500,S10,Midterm 2


In [25]:
df = pd.concat([eda, hr, temp], axis=1).drop(columns=['Unnamed: 0'])

In [29]:
df[df['student2'] == 'S10']

,eda,student,test,hr,student1,test1,temp,student2,test2
1581984,0.014094,S9,Midterm 2,NaN,NaN,NaN,22.41,S10,Final
1581985,0.015375,S9,Midterm 2,NaN,NaN,NaN,22.41,S10,Final
1581986,0.015375,S9,Midterm 2,NaN,NaN,NaN,22.41,S10,Final
1581987,0.017938,S9,Midterm 2,NaN,NaN,NaN,22.41,S10,Final
1581988,0.014094,S9,Midterm 2,NaN,NaN,NaN,22.41,S10,Final
...,...,...,...,...,...,...,...,...,...
1772979,0.020500,S10,Midterm 2,NaN,NaN,NaN,22.95,S10,Midterm 2
1772980,0.020500,S10,Midterm 2,NaN,NaN,NaN,22.91,S10,Midterm 2
1772981,0.020500,S10,Midterm 2,NaN,NaN,NaN,22.91,S10,Midterm 2
1772982,0.019219,S10,Midterm 2,NaN,NaN,NaN,22.91,S10,Midterm 2


In [36]:
hr = pd.read_csv("data/HR_df.csv").rename(columns={'data':'hr'})
temp = pd.read_csv("data/TEMP_df.csv").drop(columns=['Unnamed: 0'])

In [37]:
hr

,hr,student,test
0,116.00,S1,Final
1,82.50,S1,Final
2,96.33,S1,Final
3,86.25,S1,Final
4,98.60,S1,Final
...,...,...,...
442997,115.07,S10,Midterm 2
442998,116.13,S10,Midterm 2
442999,116.07,S10,Midterm 2
443000,116.47,S10,Midterm 2


In [154]:
# Step 1: Aggregate features per student
eda_features = eda.groupby('student')['eda'].agg(['mean', 'std', 'min', 'max']).reset_index().rename(columns={'mean':'eda_mean','std':'eda_std','min':'eda_min','max':'eda_max'})
temp_features = temp.groupby('student')['temp'].agg(['mean', 'std', 'min', 'max']).reset_index().rename(columns={'mean':'temp_mean','std':'temp_std','min':'temp_min','max':'temp_max'})
hr_features = hr.groupby('student')['hr'].agg(['mean', 'std', 'min', 'max']).reset_index().rename(columns={'mean':'hr_mean','std':'hr_std','min':'hr_min','max':'hr_max'})

# Step 2: Merge features per student
merged_features = temp_features.merge(hr_features, on='student', how='inner').merge(eda_features, on='student', how='inner')

# Step 3: Remove 'student' column for clustering (or keep as index)
X = merged_features.drop(columns=['student'])

# Step 4: Fit KMeans
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans.fit(X)

# Optional: Add cluster labels
merged_features['cluster'] = kmeans.labels_

In [155]:
merged_features

,student,temp_mean,temp_std,temp_min,temp_max,hr_mean,hr_std,hr_min,hr_max,eda_mean,eda_std,eda_min,eda_max,cluster
0,S1,26.471800,3.857407,16.25,32.73,106.573052,21.928012,57.00,194.47,0.160783,0.284260,0.0,2.863800,0
1,S10,26.182542,4.080916,17.23,32.18,106.274096,27.105398,49.00,191.70,0.149010,0.185994,0.0,7.412322,2
2,S2,28.218365,5.682163,15.19,36.07,114.334012,25.550614,57.33,206.12,0.111307,0.175202,0.0,2.146993,1
3,S3,26.897273,4.365469,13.89,33.25,103.301189,24.085842,54.08,196.88,0.271979,0.372477,0.0,5.054469,0
4,S4,26.821141,4.049378,14.99,32.43,98.491765,30.031735,56.45,203.17,0.463462,0.707830,0.0,6.693083,1
5,S5,28.551023,4.705315,13.91,34.00,104.813908,22.220718,50.00,205.02,0.370447,0.348536,0.0,3.671499,1
6,S6,27.224615,4.916618,14.83,34.34,104.821049,28.802255,54.03,204.57,0.256247,0.398971,0.0,3.935066,1
7,S7,25.566760,2.969450,15.99,30.21,103.876402,25.742649,54.88,194.68,0.207661,0.304124,0.0,3.340750,0
8,S8,28.462105,4.798693,15.01,34.79,105.516276,26.863789,51.00,200.33,0.392806,0.498147,0.0,3.789544,1
9,S9,28.443362,5.162969,12.51,34.66,98.872328,23.626482,56.75,205.53,0.298770,0.346178,0.0,3.924276,1


In [223]:
# import joblib
# joblib.dump(kmeans, 'kmeans_model.pkl')

In [156]:
kmeans.predict(X)

array([0, 2, 1, 0, 1, 1, 1, 0, 1, 1], dtype=int32)

In [157]:
int(kmeans.predict(predict)[0])

0

In [94]:
cols = list(X.columns)

In [117]:
cols

Index(['temp_mean', 'temp_std', 'temp_min', 'temp_max', 'hr_mean', 'hr_std',
       'hr_min', 'hr_max', 'eda_mean', 'eda_std', 'eda_min', 'eda_max'],
      dtype='object')

In [116]:
X.mean()

temp_mean     27.283899
temp_std       4.458838
temp_min      14.980000
temp_max      33.466000
hr_mean      104.687408
hr_std        25.595749
hr_min        54.052000
hr_max       200.247000
eda_mean       0.268247
eda_std        0.362172
eda_min        0.000000
eda_max        4.283180
dtype: float64

In [153]:
round(np.random.choice(np.arange(0.55,0.85,0.01)), 2)

np.float64(0.64)

In [176]:
def predictVals(temp,hr,eda):
    input_df = pd.DataFrame(columns=['temp_mean', 'temp_std', 'temp_min', 'temp_max', 'hr_mean', 'hr_std',
       'hr_min', 'hr_max', 'eda_mean', 'eda_std', 'eda_min', 'eda_max'])
    new_row = pd.DataFrame([temp, 4.458838, 14.980000, 33.466000, hr, 25.595749, 54.052000, 200.247000, eda, 0.362172, 0, 4.283180],index=cols).T
    predict = pd.concat([input_df,new_row], ignore_index=True)
    p = int(kmeans.predict(predict)[0])
    print(p)
    ## high stress
    if p == 0:
        return np.random.choice(np.arange(0.6,0.8,0.01))
    ## medium stress
    elif p == 1:
        return np.random.choice(np.arange(0.75,0.95,0.01))
    ## low stress
    else:
        return np.random.choice(np.arange(0.5,0.75,0.01))

In [222]:
predictVals(45,200,0.10)

2


/var/folders/y6/yynhnkp53m70dw4wkft15lbw0000gn/T/ipykernel_24110/2062320805.py:5: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  predict = pd.concat([input_df,new_row], ignore_index=True)


np.float64(0.7200000000000002)

In [104]:
input_df = pd.DataFrame(columns=cols)

In [110]:
new_row = pd.DataFrame([26,4,16,32,106,22,57,195,0.2,0.4,0,2.9],index=cols).T

In [112]:
predict = pd.concat([input_df,new_row], ignore_index=True)

/var/folders/y6/yynhnkp53m70dw4wkft15lbw0000gn/T/ipykernel_24110/4003420223.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  predict = pd.concat([input_df,new_row], ignore_index=True)


In [66]:
grades.set_index("Unnamed: 0").apply('mean', axis=1)

Unnamed: 0
S1     0.836667
S2     0.856667
S3     0.870000
S4     0.753333
S5     0.740000
S6     0.743333
S7     0.506667
S8     0.906667
S9     0.606667
S10    0.703333
dtype: float64

In [19]:
df.groupby('student')[['eda','hr','temp']].mean()

,eda,hr,temp
student,,,
S1,0.160783,105.786614,26.471830
S10,0.149010,NaN,26.184913
S2,0.111307,103.916531,28.218354
S3,0.271979,105.989287,26.897198
S4,0.463462,NaN,26.820075
S5,0.370447,NaN,28.549950
S6,0.256247,NaN,27.223881
S7,0.207661,NaN,25.566456
S8,0.392806,NaN,28.461554


In [130]:
hr_mean = df['hr'].mean()
temp_mean = df['temp (C°)'].mean()
df['hr'] = df['hr'].fillna(hr_mean)
df['temp (C°)'] = df['temp (C°)'].fillna(temp_mean)

In [132]:
df.set_index('student')

,eda,hr,temp (C°)
student,,,
S1,0.000000,116.000000,21.890000
S1,0.005125,82.500000,21.890000
S1,0.020501,96.330000,21.890000
S1,0.021783,86.250000,21.890000
S1,0.023064,98.600000,21.890000
...,...,...,...
S10,0.021781,104.947328,27.249751
S10,0.020500,104.947328,27.249751
S10,0.020500,104.947328,27.249751


In [49]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [122]:
df_features = list(df.columns)
X = df.loc[: , df_features].values
X = StandardScaler().fit_transform(X)

In [123]:
X

array([[-6.53349064e-01,  8.47134098e-01, -1.15531498e+00],
       [-6.40474728e-01, -1.72047962e+00, -1.15531498e+00],
       [-6.01849207e-01, -6.60476705e-01, -1.15531498e+00],
       ...,
       [-6.01851719e-01,  1.41595168e-14,  0.00000000e+00],
       [-6.01851719e-01,  1.41595168e-14,  0.00000000e+00],
       [-5.98633763e-01,  1.41595168e-14,  0.00000000e+00]])

In [127]:
df_features

['eda', 'hr', 'temp (C°)']

In [138]:
df

,eda,student,hr,temp (C°)
0,0.000000,S1,116.000000,21.890000
1,0.005125,S1,82.500000,21.890000
2,0.020501,S1,96.330000,21.890000
3,0.021783,S1,86.250000,21.890000
4,0.023064,S1,98.600000,21.890000
...,...,...,...,...
1773115,0.021781,S10,104.947328,27.249751
1773116,0.020500,S10,104.947328,27.249751
1773117,0.020500,S10,104.947328,27.249751
1773118,0.020500,S10,104.947328,27.249751


In [86]:
cluster1 = hr[(hr['student'] == 'S1') | (hr['student'] == 'S2') | (hr['student'] == 'S3')]
cluster2 = hr[(hr['student'] == 'S4') | (hr['student'] == 'S5') | (hr['student'] == 'S6') | (hr['student'] == 'S8') | (hr['student'] == 'S9')]
cluster3 = hr[(hr['student'] == 'S10') | (hr['student'] == 'S7')]

In [87]:
cluster1.set_index('student')['hr'].mean()

np.float64(108.22225091310041)

In [88]:
cluster2.set_index('student')['hr'].mean()

np.float64(102.57121740430478)

In [89]:
cluster3.set_index('student')['hr'].mean()

np.float64(105.14022233021505)

In [143]:
cluster1.set_index('student').mean()

eda            0.180309
hr           104.947328
temp (C°)     27.229207
dtype: float64

In [144]:
cluster2.set_index('student').mean()

eda            0.352295
hr           104.947328
temp (C°)     27.856603
dtype: float64

In [145]:
cluster3.set_index('student').mean()

eda            0.176748
hr           104.947328
temp (C°)     25.892826
dtype: float64

In [152]:
df1 = grades.set_index("Unnamed: 0")

In [157]:
cluster1_grade = df1[(df1.index == 'S1') | (df1.index == 'S2') | (df1.index == 'S3')].mean().mean()

In [160]:
cluster2_grade = df1[(df1.index == 'S4') | (df1.index == 'S5') | (df1.index == 'S6') | (df1.index == 'S8') | (df1.index == 'S9')].mean().mean()

In [162]:
cluster3_grade = df1[(df1.index == 'S10') | (df1.index == 'S7')].mean().mean()

In [164]:
cluster1_grade

np.float64(0.8544444444444443)

In [165]:
cluster2_grade

np.float64(0.75)

In [163]:
cluster3_grade

np.float64(0.605)

In [139]:
grouped = df.groupby('student')

# Compute summary features per student
features = grouped.agg(['mean', 'std', 'min', 'max', 'median'])

# Flatten the MultiIndex columns
features.columns = ['_'.join(col) for col in features.columns]

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

# Cluster
kmeans = KMeans(n_clusters=3, random_state=42)
labels = kmeans.fit_predict(X_scaled)

# Add cluster labels to the DataFrame
features['cluster'] = labels

print(features[['cluster']])

         cluster
student         
S1             0
S10            2
S2             0
S3             0
S4             1
S5             1
S6             1
S7             2
S8             1
S9             1


In [128]:
grades.set_index("Unnamed: 0").apply('mean', axis=1)

Unnamed: 0
S1     0.836667
S2     0.856667
S3     0.870000
S4     0.753333
S5     0.740000
S6     0.743333
S7     0.506667
S8     0.906667
S9     0.606667
S10    0.703333
dtype: float64

In [59]:
df1

,eda,student,test,hr,temp (C°)
0,0.000000,S1,Final,116.00,21.89
1,0.005125,S1,Final,82.50,21.89
2,0.020501,S1,Final,96.33,21.89
3,0.021783,S1,Final,86.25,21.89
4,0.023064,S1,Final,98.60,21.89
...,...,...,...,...,...
1773115,0.021781,S10,Midterm 2,NaN,NaN
1773116,0.020500,S10,Midterm 2,NaN,NaN
1773117,0.020500,S10,Midterm 2,NaN,NaN
1773118,0.020500,S10,Midterm 2,NaN,NaN


In [28]:
hr

,data,student,test
0,116.00,S1,Final
1,82.50,S1,Final
2,96.33,S1,Final
3,86.25,S1,Final
4,98.60,S1,Final
...,...,...,...
442997,115.07,S10,Midterm 2
442998,116.13,S10,Midterm 2
442999,116.07,S10,Midterm 2
443000,116.47,S10,Midterm 2


In [5]:
hr = pd.read_csv("data/HR_df.csv").rename(columns={'data':'hr', 'student':'student1','test':'test1'})

In [54]:
hr

,hr,student1,test1
0,116.00,S1,Final
1,82.50,S1,Final
2,96.33,S1,Final
3,86.25,S1,Final
4,98.60,S1,Final
...,...,...,...
442997,115.07,S10,Midterm 2
442998,116.13,S10,Midterm 2
442999,116.07,S10,Midterm 2
443000,116.47,S10,Midterm 2


In [26]:
hr.groupby('student')['data'].mean()

student
S1     106.573052
S10    106.274096
S2     114.334012
S3     103.301189
S4      98.491765
S5     104.813908
S6     104.821049
S7     103.876402
S8     105.516276
S9      98.872328
Name: data, dtype: float64

In [24]:
eda.groupby('student')['data'].mean()

student
S1     0.160783
S10    0.149010
S2     0.111307
S3     0.271979
S4     0.463462
S5     0.370447
S6     0.256247
S7     0.207661
S8     0.392806
S9     0.298770
Name: data, dtype: float64

In [22]:
grades.set_index("Unnamed: 0").apply('mean', axis=1)

Unnamed: 0
S1     0.836667
S2     0.856667
S3     0.870000
S4     0.753333
S5     0.740000
S6     0.743333
S7     0.506667
S8     0.906667
S9     0.606667
S10    0.703333
dtype: float64

In [6]:
hr = pd.read_csv("data/HR_df.csv").rename(columns={'data':'hr', 'student':'student1','test':'test1'})
temp = pd.read_csv("data/TEMP_df.csv").rename(columns={'data':'temp','student':'student2','test':'test2'})

In [56]:
temp

,temp (C°),student2,test2
0,21.89,S1,Final
1,21.89,S1,Final
2,21.89,S1,Final
3,21.89,S1,Final
4,21.89,S1,Final
...,...,...,...
1772979,22.95,S10,Midterm 2
1772980,22.91,S10,Midterm 2
1772981,22.91,S10,Midterm 2
1772982,22.91,S10,Midterm 2


In [7]:
t1 = temp[temp['test'] == 'Final']

In [9]:
t1.groupby('student')['temp (C°)'].mean()

student
S1     26.184411
S10    26.331471
S2     28.182078
S3     26.509118
S4     28.376879
S5     30.185718
S6     27.047806
S7     25.964929
S8     29.274745
S9     30.808438
Name: temp (C°), dtype: float64

In [8]:
grades = pd.read_csv("data/grades_df.csv")

In [9]:
grades.set_index("Unnamed: 0").apply('mean', axis=1)

Unnamed: 0
S1     0.836667
S2     0.856667
S3     0.870000
S4     0.753333
S5     0.740000
S6     0.743333
S7     0.506667
S8     0.906667
S9     0.606667
S10    0.703333
dtype: float64